In [ ]:
import os
import sounddevice as sd
import numpy as np
from faster_whisper import WhisperModel
import ollama
from IPython.display import clear_output
import time
from pydantic import BaseModel
import time

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from buffer import AudioBuffer
from transcription_pipeline import TranscriptionPipeline
from mergestrat import MergeStrategySelector

In [ ]:
print(sd.query_devices())

In [ ]:
whisper_model = WhisperModel(model_size, device="cuda")
audio_buffer = AudioBuffer(window_size = 5, overlap_size = 2)
merge_selector = MergeStrategySelector(context_words = 10)
pipeline = TranscriptionPipeline(whisper_model, merge_selector)

def audio_callback(indata, frames, times, status):
    audio_buffer.append_chunk(indata)
    
with sd.InputStream(samplerate = 16000, channels = 1, callback = audio_callback, blocksize=16000):
    while True:
        sd.sleep(100)
        window = audio_buffer.get_window()
        if window is None:
            continue
        transcribed_text = pipeline.process_and_merge(window)
        print(audio_buffer.get_buffer_size())
        print(transcribed_text)